# Step 5: Feature Engineering & Preprocessing Pipeline
Data Scientist Intern Portfolio Project
OBJECTIVE - Transform categorical text columns into mathematically optimized numerical vectors using One-Hot Encoding and Ordinal Mapping to make the dataset completely model-ready.
Transform categorical text features into mathematically optimized numerical vectors. Machine learning models operate on pure linear algebra and cannot process raw strings; this notebook builds the mathematical bridge



In [7]:
import pandas as pd
import numpy as np

# Load our clean dataset from our processed folder
DATA_PATH = r"C:\Users\Admin\Desktop\Finance-credit-risk\data\processed\credit_risk_clean.csv"
df = pd.read_csv(DATA_PATH)

print(f"✅ Clean data loaded successfully for feature engineering! Rows: {df.shape[0]}")
# Let's inspect the text columns we need to transform
print("\nText columns to process:")
print(df.select_dtypes(include=['object']).columns.tolist())


✅ Clean data loaded successfully for feature engineering! Rows: 31679

Text columns to process:
['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']


In [8]:
# 1. Create a dictionary to map loan grades to numbers (A is safest = 0, G is riskiest = 6)
grade_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}
df['loan_grade_encoded'] = df['loan_grade'].map(grade_mapping)

# 2. Map the historical default flag (N = No historical default = 0, Y = Yes historical default = 1)
default_mapping = {'N': 0, 'Y': 1}
df['cb_historical_default_encoded'] = df['cb_person_default_on_file'].map(default_mapping)

# Let's preview our new numeric columns right next to the original text columns
print("--- Check Ordinal Mapping Results ---")
print(df[['loan_grade', 'loan_grade_encoded', 'cb_person_default_on_file', 'cb_historical_default_encoded']].head())


--- Check Ordinal Mapping Results ---
  loan_grade  loan_grade_encoded cb_person_default_on_file  \
0          B                   1                         N   
1          C                   2                         N   
2          C                   2                         N   
3          C                   2                         Y   
4          A                   0                         N   

   cb_historical_default_encoded  
0                              0  
1                              0  
2                              0  
3                              1  
4                              0  


### Step 2: Algorithmic Text Encoding (Ordinal vs. Nominal)
Objective- Apply distinct data-encoding techniques based on the financial and logical nature of each text column.

1.  Ordinal Mapping (`loan_grade` & `cb_person_default_on_file`): These columns contain natural risk-based hierarchies. We manually map them to ordered integers (e.g., Credit Grade A = 0, G = 6; Historical Default No = 0, Yes = 1) to ensure the machine learning model respects their built-in scale.
2.  One-Hot Encoding (`loan_intent`):** Loan purposes (like Medical vs. Education) have no inherent numerical rank or order. We utilize `pd.get_dummies()` to break this column into separate binary indicators (0 or 1), preventing the model from assuming an artificial mathematical hierarchy between categories.


In [9]:
# Use get_dummies to split 'loan_intent' into separate 1 and 0 light-switch columns
# prefix='intent' ensures our new columns look neat (e.g., intent_MEDICAL, intent_VENTURE)
intent_dummies = pd.get_dummies(df['loan_intent'], prefix='intent', dtype=int)

# Combine these brand new columns back into our main dataset (df)
df = pd.concat([df, intent_dummies], axis=1)

# Let's print out all the column names to see our new automated features!
print("--- Updated Dataset Columns ---")
print(df.columns.tolist())


--- Updated Dataset Columns ---
['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_status', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length', 'loan_grade_encoded', 'cb_historical_default_encoded', 'intent_DEBTCONSOLIDATION', 'intent_EDUCATION', 'intent_HOMEIMPROVEMENT', 'intent_MEDICAL', 'intent_PERSONAL', 'intent_VENTURE']


In [4]:
# 1. Establish the collateral scale mapping dictionary
home_mapping = {'OTHER': 0, 'RENT': 1, 'MORTGAGE': 2, 'OWN': 3}

# 2. Apply it directly to the dataframe column
df['home_ownership_encoded'] = df['person_home_ownership'].map(home_mapping)

# 3. Quickly verify no NaN anomalies were introduced
print("Unique values in new column:", df['home_ownership_encoded'].unique())


Unique values in new column: [3 2 1 0]


### Step 3: Collateral Mapping & Dimensionality Cleanup
Objective: Process the final structural text column and strip the dataset of redundant raw variables to finalize the machine learning matrix.

1.  Collateral Scale Mapping: We map `person_home_ownership` into a 0-3 logical index representing asset security (`RENT` = 1, `MORTGAGE` = 2, `OWN` = 3), providing the algorithm with a clear proxy for borrower collateral.
2.  Redundant Column Dropping: The raw text features are now fully captured by our numeric vectors. We remove the original string columns to prevent model crashes and eliminate data redundancy.

 Resulting Matrix Integrity: The dataset successfully expands to **17 dimensions of pure numeric values, completely structured for downstream classification.


In [10]:
# List out the old redundant text columns to drop
columns_to_drop = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

# Drop them along the vertical column axis (axis=1)
df_final = df.drop(columns=columns_to_drop)

print("--- FINAL PREPROCESSED DATASET INTEGRITY CHECK ---")
print(f"Total Columns Remaining: {len(df_final.columns)}")
print("\nFinal column names to feed directly into the machine learning model:")
print(df_final.columns.tolist())


--- FINAL PREPROCESSED DATASET INTEGRITY CHECK ---
Total Columns Remaining: 16

Final column names to feed directly into the machine learning model:
['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_status', 'loan_percent_income', 'cb_person_cred_hist_length', 'loan_grade_encoded', 'cb_historical_default_encoded', 'intent_DEBTCONSOLIDATION', 'intent_EDUCATION', 'intent_HOMEIMPROVEMENT', 'intent_MEDICAL', 'intent_PERSONAL', 'intent_VENTURE']
